# 07 — Efficient Attention — Solution

---

In [ ]:
import sys, os, math, time
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from src.models.attention import MultiHeadAttention, scaled_dot_product_attention, create_causal_mask
from src.models.efficient_attention import SlidingWindowAttention, LinearAttention

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — Quadratic Memory Scaling

In [ ]:
def measure_attention_memory(seq_len, d_model=128, n_heads=4):
    if not torch.cuda.is_available():
        return n_heads * seq_len * seq_len * 4 / 1e6
    torch.cuda.reset_peak_memory_stats()
    attn = MultiHeadAttention(d_model, n_heads).to('cuda')
    x = torch.randn(1, seq_len, d_model, device='cuda')
    with torch.no_grad(): attn(x, x, x)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 1e6

seq_lengths = [64, 128, 256, 512, 1024]
memories    = [measure_attention_memory(s) for s in seq_lengths]

# O(n^2) reference line
ref_scale = memories[0] / (seq_lengths[0] ** 2)
ref_line  = [ref_scale * s**2 for s in seq_lengths]

plt.figure(figsize=(8, 4))
plt.plot(seq_lengths, memories, 'o-', color='tomato',  label='Measured')
plt.plot(seq_lengths, ref_line, '--', color='gray',    label='O(n²) reference')
plt.xlabel('Sequence length'); plt.ylabel('Memory (MB)')
plt.title('Attention Memory Scaling'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

base_mem = memories[-1]  # at seq=1024
base_seq = 1024
print('\nExtrapolated memory (O(n²) scaling):')
for s in [2048, 8192, 32768]:
    est = base_mem * (s / base_seq) ** 2
    print(f'  seq={s:6d}: ~{est:.0f} MB')

## Part B — Sliding Window Mask

In [ ]:
def make_sliding_window_mask(seq_len, window_size, causal=True):
    rows = torch.arange(seq_len).unsqueeze(1)
    cols = torch.arange(seq_len).unsqueeze(0)
    dist = torch.abs(rows - cols)
    mask = (dist <= window_size).float()   # 1 = can attend
    if causal:
        causal_mask = (cols <= rows).float()
        mask = mask * causal_mask
    return mask.view(1, 1, seq_len, seq_len)


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, causal, title in zip(axes, [False, True], ['Bidirectional (window=3)', 'Causal (window=3)']):
    m = make_sliding_window_mask(12, 3, causal=causal)
    ax.imshow(m.squeeze(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title); ax.set_xlabel('Key pos'); ax.set_ylabel('Query pos')
plt.tight_layout(); plt.show()

# Memory comparison
seq, window = 256, 32
print(f'\nAttention elements (seq={seq}, window={window}):')
print(f'  Full: {seq*seq:,}   Sliding: {seq*(2*window+1):,}   ({seq*seq/(seq*(2*window+1)):.1f}x reduction)')

## Parts C & D — Library + Linear Attention

In [ ]:
swa = SlidingWindowAttention(d_model=128, n_heads=4, window_size=32)
linear_attn = LinearAttention(d_model=128, n_heads=4)
x = torch.randn(2, 128, 128)

print('SlidingWindowAttention:', swa(x)[0].shape)
print('LinearAttention       :', linear_attn(x)[0].shape)

if hasattr(F, 'scaled_dot_product_attention'):
    B, H, S, dk = 2, 4, 64, 32
    q = torch.randn(B, H, S, dk, device=device)
    k = torch.randn(B, H, S, dk, device=device)
    v = torch.randn(B, H, S, dk, device=device)
    out_sdpa = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    mask = create_causal_mask(S, device=device)
    out_manual, _ = scaled_dot_product_attention(q, k, v, mask=mask)
    print(f'\nPyTorch SDPA == manual: {torch.allclose(out_sdpa, out_manual, atol=1e-4)}')

print('\nComplexity summary:')
rows = [
    ('Standard',       'O(n²d)',    'Yes',   'All transformers'),
    ('Sliding Window', 'O(nwd)',    'Yes',   'Mistral, Longformer'),
    ('FlashAttention', 'O(n²)',     'Yes',   'LLaMA, Gemma'),
    ('Linear',         'O(nd²)',    'Approx','Research models'),
]
for name, comp, exact, used in rows:
    print(f'  {name:<18} {comp:<10} exact={exact:<6} used in: {used}')